# 🥈 Silver Layer — Clean and Validate Loan Transactions

**What we're doing:** Taking Bronze raw data and cleaning it.
- Remove failed/invalid transactions
- Standardize date formats
- Add useful columns for reporting

In [ ]:
# Import PySpark SQL helper functions used for Silver-layer cleansing and column transforms
from pyspark.sql import functions as F

# Read the raw Bronze Delta table so the Silver layer starts from the ingested source data
df = spark.table('bronze_loan_transactions')
# Count the Bronze input records before applying Silver filtering and type corrections
print(f'Bronze records: {df.count()}')

In [ ]:
# Keep only completed loan transactions so the Silver layer excludes unfinished raw events
# Convert TransactionDate to a real date, derive a month key, and cast Amount to double for analysis
df_silver = df \
    .filter(F.col('Status') == 'Completed') \
    .withColumn('TransactionDate', F.to_date('TransactionDate', 'yyyy-MM-dd')) \
    .withColumn('Month', F.date_format('TransactionDate', 'yyyy-MM')) \
    .withColumn('Amount', F.col('Amount').cast('double'))

# Count the cleaned Silver rows to confirm how many completed records remain after filtering
print(f'✅ Silver records after cleaning: {df_silver.count()}')
# Preview the cleaned Silver dataset to verify the transformed columns look correct
df_silver.show(10)

In [ ]:
# Write the cleaned Silver DataFrame to a Delta table for downstream Gold reporting
df_silver.write.format('delta').mode('overwrite').saveAsTable('silver_loan_transactions')
# Confirm that the Silver Delta table is ready for aggregated reporting jobs
print('✅ Silver table saved!')

In [ ]:
%%sql
-- Summarize cleaned Silver transactions by type to validate counts and total loan volume
SELECT TransactionType, COUNT(*) as Count, ROUND(SUM(Amount),2) as TotalAmount
FROM silver_loan_transactions
GROUP BY TransactionType